# Data Mining / Prospecção de Dados

## Sara C. Madeira, 2025/2026

# Project 1 - Pattern Mining

## Logistics 


**The project's solution should be uploaded in Moodle before the end of `May, 10th (23:59)`.** 

Students should **upload a `.zip` file** containing a folder with all the files necessary for project evaluation. 
Groups should be registered in [Moodle](https://moodle.ciencias.ulisboa.pt/mod/groupselect/view.php?id=299650) and the `zip` file should be identified as **`PDnn.zip`** where `nn` is the number of your group.

**It is mandatory to produce a Jupyter notebook containing code and text/images/tables/etc describing the solution and the results. Projects not delivered in this format will not be graded. You can use `PD_202526_P1.ipynb` as template. In your `PDnn.zip` folder you should also include an `HTML` version of your notebook with all the outputs.**

**Decisions should be justified and results should be critically discussed.** 

Remember that **only the relevant code and experiments should be presented, not everything you tried and did not work, or is not relevant** (that can be discussed in the text, if relevant)! Tables and figures can be used together with text to summarize results and conclusions, improving understanding, readability and concision.

_**Project solutions containing only code and outputs without discussions will achieve a maximum grade of 10 out of 20.**_

## Team Identification

**GROUP 08**

Students:

* Valentin Barner - 66958
* Jonas Auffermann - 66939
* Rocco Cocre - 

## Dataset and Tools

In this project you will analyse `data from an online Store` collected over 4 months (April - July) and stored in `3 files`:

1. `store-products.csv` - contains the list of products sold by the online store (**46.294 different products** associated with **123 different subcategories**). Each record/line in the file has the following fields:

* **Item ID** -  unique identifier of the product. 
* **Product Categories** - category and subcategories of the product, represented as a string containing the category and subcategories of the item. Example: in `appliances.kitchen.juice`, `appliances` is the category, `kitchen` is the subcategory and `juice` is the product.

2. `store-clicks.csv` - contains click events (relating users and product clicks) identified by a session ID (**5.613.499 sessions**). Each record/line in the file has the following fields (with this order):

* **Session ID** – id of the session. In one session there are one or many clicks. Could be represented as an integer number.
* **Timestamp** – time when the click occurred. Format of YYYY-MM-DDThh:mm:ss.SSSZ
* **Item ID** – unique identifier of the product that was clicked. 
* **Context** – context of the click. The value "S" indicates a special offer, "0" indicates a missing value, a number between 1 to 12 indicates a real category identifier,
any other number indicates a brand. For example if a product was clicked in the context of a promotion or special offer then the value will be "S", if the context was a brand i.e BOSCH,
then the value will be an 8-10 digits number. If the item was clicked under regular category, that is sport, then the value will be a number between 1 to 12.

3. `store-purchases.csv` - contains purchase events (relating users and product purchases) identified by a session ID (**318.444 sessions**). Each record/line in the file has the following fields (with this order): 

* **Session ID** - id of the session. In one session there are one or many purchases.
* **Timestamp** - time when the purchase occurred (Format: YYYY-MM-DDThh:mm:ss.SSSZ)
* **Item ID** – unique identifier of the product that was bought.  
* **Price** –  price of the product. 
* **Quantity** – quantity purchased.

Your **goal** is to find **actionable patterns** by `Mining Frequent Patterns and Association Rules`. Specifically, you should mine clicks and purchases separately and together/merged, to identify click patterns and purchase patterns, together with click-purchase patterns, respectively. Furthermore, you should try to unravel differences in `week` versus `weekend`patterns.

In this context, the project has **2 main tasks**:

1. Unravelling Click, Purchase and Click-Purchase Patterns and Rules **(global patterns)**
   
3. Looking for Differences between `Week` and `Weekend`Patterns and Rules **(local/specific patterns)**

**While doing PATTERN and ASSOCIATION RULE MINING keep in mind the following basic/key questions and BE CREATIVE!**

1. What are the most popular products?
2. What are the most purchased products?
3. Which products are clicked together (click patterns)? Which products are purchased together (purchase patterns)?
4. What about products that are clicked and actually purchased together (click-purchase patterns)?
5. Can we find associations highlighting that when people buy a product/set of products also buy other product(s)?
6. Can we find associations highlighting that when people click a product/set of products actually buy these product(s)? Or do they buy other product(s)?
7. Are these associations strong? Can we trust them? Are they misleading?
9. Can we analyse these patterns and evaluate these associations to find, not only frequent and strong associations, but also interesting patterns and associations that can become actionable patterns?
8. What can we say about the patterns and association rules found during `week` sessions versus `weekend`sessions? Are there differences?

**In this project you should use [Python 3](https://www.python.org), [Jupyter Notebook](http://jupyter.org) and [`MLxtend`](http://rasbt.github.io/mlxtend/)/other libraries used in the TP lessons.**

**Choose the pattern mining algorithm to be used.** 

## 1. Unravelling Click, Purchase and Click-Purchase Patterns and Rules

In this first task you should load and preprocess data in order to compute frequent itemsets and generate association rules targeting:

    - Click patterns
    - Purchase patterns
    - Click-purchase patterns

### 1.1. Load and Preprocess Data

 **Product quantities and other irrelevant info for the pattern mining tasks should not be considered.**

In [ ]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules
from collections import Counter

def one_hot_encode(transactions):
    te = TransactionEncoder()
    te_ary = te.fit(transactions).transform(transactions, sparse=True)
    return pd.DataFrame.sparse.from_spmatrix(te_ary, columns=te.columns_)

def filter_transactions(transactions, min_support_abs):
    item_counts = Counter(item for transaction in transactions for item in transaction)
    frequent_items = {item for item, count in item_counts.items() if count >= min_support_abs}
    filtered_transactions = [[item for item in transaction if item in frequent_items] for transaction in transactions]
    return filtered_transactions

products_df = pd.read_csv('datasets/store-products.csv', header=None)


In [ ]:



# load the datasets
# session id, timestamp, item id, context
clicks_df = pd.read_csv('datasets/store-clicks.csv', header=None)

# item id, product categories
products_df = pd.read_csv('datasets/store-products.csv', header=None)

# session id, timestamp, item id, price, quantity
purchases_df = pd.read_csv('datasets/store-purchases.csv', header=None)

# only keep session id and item id for transactions
clicks = clicks_df.iloc[:, [0, 2]]
purchases = purchases_df.iloc[:, [0, 2]]

click_transactions = clicks.groupby(0)[2].apply(list).tolist()
purchase_transactions = purchases.groupby(0)[2].apply(list).tolist()

# free memory by deleting the original dataframes
del clicks_df, purchases_df

purchase_transactions = filter_transactions(purchase_transactions, 3)
click_transactions = filter_transactions(click_transactions, 3)

# print("Purchase transactions: ", purchase_transactions[:5])
print("Click transactions: ", click_transactions[:5])

purchase_encoded = one_hot_encode(purchase_transactions)
click_encoded = one_hot_encode(click_transactions)

# columns are item ids, rows are transactions, values are 1 if item is in transaction, 0 otherwise


C:\Users\Valentin\AppData\Local\Temp\ipykernel_19504\3683519711.py:20: DtypeWarning: Columns (0: 3) have mixed types. Specify dtype option on import or set low_memory=False.
  clicks_df = pd.read_csv('datasets/store-clicks.csv', header=None)


Click transactions:  [[214536502, 214536500, 214536506, 214577561], [214662742, 214662742, 214825110, 214757390, 214757407, 214551617], [214716935, 214774687, 214832672], [214836765, 214706482], [214701242, 214826623]]


In [ ]:
# combine clicks and purchases for click-purchase patterns by merging on session id and item id, then one-hot encode the combined transactions
combined_transactions = pd.merge(clicks_df.iloc[:, [0, 2]], purchases_df.iloc[:, [0, 2]], on=[0, 2], how='outer')
combined_transactions = combined_transactions.groupby(0)[2].apply(list).tolist()


In [3]:
click_encoded.head()

,214507224,214507226,214507239,214507331,214507365,214507385,214507387,214507408,214507415,214507445,...,643078950,1178794852,1178812378,1178818515,1178818815,1178824421,1178829538,1178833258,1178833614,1178837797
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


Write text in cells like this ...


### 1.2. Compute Frequent Itemsets

* Compute frequent itemsets considering a minimum support S_min. 
* Present frequent itemsets organized by length (number of items). 
* List frequent k-itemsets with support of at least S < S_min.
* Change the minimum support values and discuss the results.

In [ ]:
# Most popular items in clicks and purchases

clicks = clicks.merge(products_df, on="item_id", how="left")
purchases = purchases.merge(products_df, on="item_id", how="left")

In [ ]:
# Most popular items clicked

popular_clicked = (
    clicks.groupby(["item_id", "category"])
    .size()
    .reset_index(name="click_count")
    .sort_values("click_count", ascending=False)
)

popular_clicked.head(20)

In [2]:
from mlxtend.frequent_patterns import fpgrowth



click_encoded.columns = click_encoded.columns.astype(str)
purchase_encoded.columns = purchase_encoded.columns.astype(str)

# sample from the encoded dataframes to speed up the computation
click_encoded_sample = click_encoded.sample(frac=0.1, random_state=42)
frequent_clicks = fpgrowth(
    click_encoded_sample,
    min_support=0.001,
    use_colnames=True
)

frequent_purchases = fpgrowth(
    purchase_encoded,
    min_support=0.001,
    use_colnames=True
)

In [ ]:
import pickle

# save frequent_clicks, frequent_purchases to pickle files
frequent_clicks.to_pickle('frequent_clicks.pkl')
frequent_purchases.to_pickle('frequent_purchases.pkl')

# Click Purchase Patterns

In [7]:
click_sets = (
    clicks.groupby(0)[2]
    .apply(lambda x: set(x))
)

purchase_sets = (
    purchases.groupby(0)[2]
    .apply(lambda x: set(x))
)

In [8]:
common_sessions = click_sets.index.intersection(purchase_sets.index)

click_sets = click_sets.loc[common_sessions]
purchase_sets = purchase_sets.loc[common_sessions]

In [9]:
transactions = pd.DataFrame({
    "clicks": click_sets,
    "purchases": purchase_sets
})

In [10]:
def encode_transaction(row):
    items = []
    
    # Klicks als Präfix
    items += [f"C_{item}" for item in row["clicks"]]
    
    # Käufe als Präfix
    items += [f"P_{item}" for item in row["purchases"]]
    
    return items

transactions["items"] = transactions.apply(encode_transaction, axis=1)

In [11]:
from mlxtend.preprocessing import TransactionEncoder

te = TransactionEncoder()
te_array = te.fit(transactions["items"]).transform(transactions["items"])

df = pd.DataFrame(te_array, columns=te.columns_)

In [12]:
# Store the encoded transactions in a pickle file
df.to_pickle('encoded_transactions.pkl')

In [ ]:
# Load the encoded transactions from the pickle file
df = pd.read_pickle('encoded_transactions.pkl')

In [ ]:
from mlxtend.frequent_patterns import fpgrowth, association_rules

frequent_itemsets = fpgrowth(df, min_support=0.002, use_colnames=True)

rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.2)

In [15]:
def is_valid_rule(row):
    return (
        all(item.startswith("C_") for item in row["antecedents"]) and
        all(item.startswith("P_") for item in row["consequents"])
    )

rules = rules[rules.apply(is_valid_rule, axis=1)]

In [16]:
for _, row in rules.iterrows():
    antecedent = ", ".join(row["antecedents"])
    consequent = ", ".join(row["consequents"])
    
    print(f"{antecedent} -> {consequent} "
          f"(support={row['support']:.4f}, "
          f"confidence={row['confidence']:.2f}, "
          f"lift={row['lift']:.2f})")

C_214820392 -> P_214820392 (support=0.0029, confidence=0.47, lift=161.24)
C_214840483 -> P_214840483 (support=0.0023, confidence=0.76, lift=330.68)
C_214748293 -> P_214748293 (support=0.0021, confidence=0.68, lift=323.95)
C_214820231 -> P_214820231 (support=0.0026, confidence=0.59, lift=231.93)
C_214821277 -> P_214821277 (support=0.0093, confidence=0.76, lift=81.78)
C_214684513 -> P_214684513 (support=0.0034, confidence=0.67, lift=194.53)
C_214821285 -> P_214821285 (support=0.0047, confidence=0.73, lift=157.57)
C_214826705 -> P_214826705 (support=0.0033, confidence=0.67, lift=204.79)
C_214821290 -> P_214821290 (support=0.0032, confidence=0.69, lift=218.71)
C_214826801 -> P_214826801 (support=0.0021, confidence=0.62, lift=300.14)
C_214840762 -> P_214840762 (support=0.0021, confidence=0.71, lift=334.15)
C_214839313 -> P_214839313 (support=0.0047, confidence=0.90, lift=189.78)
C_214826608 -> P_214826608 (support=0.0024, confidence=0.82, lift=344.26)
C_214826833 -> P_214826833 (support=0.0

Write text in cells like this ...


### 1.3. Generate Association Rules from Frequent Itemsets

Using a minimum support S_min fundamented by the previous results. 
* Generate association rules with a choosed value (C) for minimum confidence. 
* Generate association rules with a choosed value (L) for minimum lift. 
* Generate association rules with both confidence >= C and lift >= L.
* Change C and L when it makes sense and discuss the results.
* Use other metrics besides confidence and lift.
* Evaluate how good the rules are given the metrics and how interesting they are from your point of view.

In [ ]:
#load frequent_clicks, frequent_purchases from pickle files
frequent_clicks = pd.read_pickle('frequent_clicks.pkl')
frequent_purchases = pd.read_pickle('frequent_purchases.pkl')

In [16]:


def compute_association_rules(frequent_itemsets):
    C = 0.5  # chosen confidence threshold
    rules_conf = association_rules(frequent_itemsets, metric="confidence", min_threshold=C)

    L = 1.2  # chosen lift threshold
    rules_lift = association_rules(frequent_itemsets, metric="lift", min_threshold=L)

    rules_both = rules_conf[rules_conf['lift'] >= L]

    return rules_both



def itemID_to_category(item_id):
    category = products_df[products_df[0] == int(item_id)][1].values
    return category[0] if len(category) > 0 else "Unknown"

def rules_to_categories(rules):
    rules['antecedent_categories'] = rules['antecedents'].apply(lambda x: [itemID_to_category(item) for item in x])
    rules['consequent_categories'] = rules['consequents'].apply(lambda x: [itemID_to_category(item) for item in x])
    return rules

def print_rules_with_categories(rules):
    # define column widths
    col1_width = 60
    col2_width = 60
    
    # header
    header = (
        f"{'Antecedent Category':<{col1_width}} | "
        f"{'Consequent Category':<{col2_width}} | "
        f"{'Antecedent IDs':<{col1_width}} | "
        f"{'Consequent IDs':<{col2_width}} | "
        f"{'Support':>8} | {'Confidence':>10} | {'Lift':>8}"
    )
    print(header)
    print("-" * len(header))
    
    # rows
    for _, row in rules.iterrows():
        antecedent_cats = ", ".join(row['antecedent_categories'])
        consequent_cats = ", ".join(row['consequent_categories'])
        antecedent_ids = ", ".join(str(item) for item in row['antecedents'])
        consequent_ids = ", ".join(str(item) for item in row['consequents'])
        print(
            f"{antecedent_cats:<{col1_width}} | "
            f"{consequent_cats:<{col2_width}} | "
            f"{antecedent_ids:<{col1_width}} | "
            f"{consequent_ids:<{col2_width}} | "
            f"{row['support']:>8.4f} | "
            f"{row['confidence']:>10.4f} | "
            f"{row['lift']:>8.4f}"
        )


In [ ]:
rules = compute_association_rules(frequent_purchases)
rules = rules_to_categories(rules)
print_rules_with_categories(rules)

Write text in cells like this ...


## Discussion of Association Rules

### Choice of Thresholds
The minimum confidence was set to 0.5, meaning that at least 50% of the transactions containing the antecedent also contain the consequent. This ensures that the generated rules are reasonably reliable.  
The minimum lift was set to 1.2 to guarantee that only rules with a positive and meaningful association are considered, filtering out rules that could occur by chance.


### Effect of Changing Confidence
Increasing the confidence threshold results in fewer rules, but those rules tend to be more reliable. However, very high confidence may eliminate potentially interesting patterns.  
Decreasing confidence increases the number of rules, but also introduces less reliable associations, making interpretation more difficult.


### Effect of Changing Lift
Increasing the lift threshold filters out rules that do not represent strong associations, keeping only those with a meaningful relationship between items.  
Lowering the lift threshold increases the number of rules but may include associations that are not truly significant and could be due to random chance.



### Using Both Confidence and Lift
Applying both constraints (confidence ≥ C and lift ≥ L) produces a smaller set of high-quality rules. These rules are both reliable and meaningful, but the number of results can become very limited depending on the thresholds chosen.


### Evaluation of Rule Quality
Good association rules typically have:
- High confidence (reliable predictions)
- Lift greater than 1 (positive association)
- Sufficient support (not too rare)

Rules with very high confidence but low lift may simply reflect very common items rather than meaningful relationships.  
Similarly, rules with very high lift but very low support may represent noise rather than useful insights.




### 1.4. Take a Look at Maximal Patterns
- discuss their utility compared to frequent patterns
- analyse the association rules they can unravel

In [ ]:
# maximal itemsets are itemsets that are not subsets of any other frequent itemset. They represent the largest combinations of items that meet the minimum support threshold.
maximal_itemsets = frequent_purchases[~frequent_purchases['itemsets'].apply(lambda x: any(set(x).issubset(set(other)) for other in frequent_purchases['itemsets'] if set(x) != set(other)))]
print("\nMaximal itemsets:")
print(maximal_itemsets)


Maximal itemsets:
      support                           itemsets
0    0.001046             frozenset({214716932})
1    0.002930             frozenset({214820392})
2    0.002292             frozenset({214840483})
3    0.001328             frozenset({214826908})
4    0.001228             frozenset({214826837})
..        ...                                ...
519  0.001181  frozenset({214844372, 214844400})
520  0.001080  frozenset({214846033, 214846029})
521  0.001809  frozenset({214846119, 214845962})
522  0.001275  frozenset({214848380, 214848337})
523  0.001539  frozenset({214716707, 214832604})

[436 rows x 2 columns]


Maximal patterns provide a compressed representation of frequent itemsets by keeping only the largest item combinations that remain frequent.
They are useful for reducing complexity and summarizing large sets of frequent patterns.
However, they discard information about subsets, which limits their usefulness for generating detailed association rules.
Association rules derived from maximal patterns are restricted to within those maximal sets and do not capture the full rule space available from all frequent itemsets.


### 1.5. Conclusions from Mining Frequent Patterns in Clicks and Purchases (Global Patterns and Rules)

### Discussion of Click Patterns
...
### Discussion of Purchase Patterns
...
### Discussion of Click-Purchase-Patterns
...

## 2. Looking for Differences between Week and Weekend Patterns and Rules

In this second part of the project the goal is to **mine `week` and `weekend` sessions separately**. 

In this context, you should try to identify specific/local patterns and associations highlighting diferences in `week` versus `weekend` behaviours.


### 2.1. Preprocess Data

**You might need to change a bit the preprocessing, although most of it should be reused.**

In [1]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules
# session id, timestamp, item id, price, quantity
purchases_df = pd.read_csv('datasets/store-purchases.csv', header=None)

# split purchases into two dfs according to timestamp for weekday / weekend
purchases_df[5] = purchases_df[1].apply(lambda x: pd.to_datetime(x).weekday())
# the days are encoded as following: 0=Monday, 1=Tuesday, ..., 4=Friday, 5=Saturday, 6=Sunday

purchases_week = purchases_df[purchases_df[5] < 5][[0,2]]
purchases_weekend = purchases_df[purchases_df[5] >= 5][[0,2]]

purchase_week_transactions = purchases_week.groupby(0)[2].apply(list).tolist()
purchase_weekend_transactions = purchases_weekend.groupby(0)[2].apply(list).tolist()




In [ ]:
# same for clicks

clicks_df = pd.read_csv('datasets/store-clicks.csv', header=None)
clicks_df[5] = clicks_df[1].apply(lambda x: pd.to_datetime(x).weekday())
clicks_week = clicks_df[clicks_df[5] < 5][[0,2]]
clicks_weekend = clicks_df[clicks_df[5] >= 5][[0,2]]
click_week_transactions = clicks_week.groupby(0)[2].apply(list).tolist()
click_weekend_transactions = clicks_weekend.groupby(0)[2].apply(list).tolist() 

C:\Users\Valentin\AppData\Local\Temp\ipykernel_25008\905717719.py:2: DtypeWarning: Columns (0: 3) have mixed types. Specify dtype option on import or set low_memory=False.
  clicks_df = pd.read_csv('datasets/store-clicks.csv', header=None)


KeyboardInterrupt: 

In [7]:
from mlxtend.frequent_patterns import fpgrowth, association_rules


# one hot encode
filtered_purchased_week_transactions = filter_transactions(purchase_week_transactions, 3)
filtered_purchased_weekend_transactions = filter_transactions(purchase_weekend_transactions, 3)

encoded_purchases_week = one_hot_encode(filtered_purchased_week_transactions)
encoded_purchases_weekend = one_hot_encode(filtered_purchased_weekend_transactions)



In [ ]:
# same for clicks
filtered_click_week_transactions = filter_transactions(click_week_transactions, 3)
filtered_click_weekend_transactions = filter_transactions(click_weekend_transactions, 3)
encoded_click_week = one_hot_encode(filtered_click_week_transactions).sample(frac=0.1, random_state=42)  # sample to speed up computation
encoded_click_weekend = one_hot_encode(filtered_click_weekend_transactions).sample(frac=0.1, random_state=42)  # sample to speed up computation

Write text in cells like this ...


### 2.2. Compute Frequent Itemsets

**This should be trivial now.**

In [13]:
encoded_purchases_week.columns = [str(i) for i in encoded_purchases_week.columns]
encoded_purchases_weekend.columns = [str(i) for i in encoded_purchases_weekend.columns]


purchases_frequent_itemsets_week = fpgrowth(
    encoded_purchases_week,
    min_support=0.001,
    use_colnames=True
)

purchases_frequent_itemsets_weekend = fpgrowth(
    encoded_purchases_weekend,
    min_support=0.001,
    use_colnames=True
)

In [ ]:

clicks_frequent_itemsets_week = fpgrowth(
    encoded_click_week,
    min_support=0.001,
    use_colnames=True
)

clicks_frequent_itemsets_weekend = fpgrowth(
    encoded_click_weekend,
    min_support=0.001,
    use_colnames=True
)

Write text in cells like this ...


### 2.3. Generate Association Rules from Frequent Itemsets

**This should be trivial now.**

In [14]:
purchases_week_rules = association_rules(purchases_frequent_itemsets_week, metric="confidence", min_threshold=0.5)
purchases_weekend_rules = association_rules(purchases_frequent_itemsets_weekend, metric="confidence", min_threshold=0.5)



In [ ]:
clicks_week_rules = association_rules(clicks_frequent_itemsets_week, metric="confidence", min_threshold=0.5)
clicks_weekend_rules = association_rules(clicks_frequent_itemsets_weekend, metric="confidence", min_threshold=0.5)

In [18]:
print_rules_with_categories(rules_to_categories(purchases_week_rules))
print_rules_with_categories(rules_to_categories(purchases_weekend_rules))

NameError: name 'products_df' is not defined

Write text in cells like this 

### 2.4.  Take a look at Maximal Patterns

**This should be trivial now.**

In [15]:
# maximal itemsets:
maximal_week_itemsets = purchases_frequent_itemsets_week[~purchases_frequent_itemsets_week['itemsets'].apply(lambda x: any(x < other for other in purchases_frequent_itemsets_week['itemsets']))]
maximal_weekend_itemsets = purchases_frequent_itemsets_weekend[~purchases_frequent_itemsets_weekend['itemsets'].apply(lambda x: any(x < other for other in purchases_frequent_itemsets_weekend['itemsets']))]

Write text in cells like this 

In [ ]:
# Write code in cells like this
# ....

Write text in cells like this


### 2.5. Conclusions from Mining Frequent Patterns in Week and Weekend (Local Patterns and Rules)

Write text in cells like this